In [2]:
import os
import numpy as np
import yaml
import json
from mozaik.storage.datastore import PickledDataStore
from parameters import ParameterSet
# from quantites import ms, Hz

In [9]:
def convert_mozaik_to_experanto(mozaik_root, output_root, bin_width_ms=33.33):
    """
    Converts Mozaik datastore to Experanto format.
    """
    # 1. Load Mozaik Data
    print("Loading Mozaik Datastore...")
    data_store = PickledDataStore(
        load=True, 
        parameters=ParameterSet({'root_directory': mozaik_root, 'store_stimuli': True})
    )
    
    # Get all segments ordered by time
    # Note: This list contains segments from ALL sheets mixed together
    all_segments = data_store.get_segments(ordered=True)
    
    # Prepare Output Directories
    responses_dir = os.path.join(output_root, 'responses')
    screen_dir = os.path.join(output_root, 'screen')
    screen_data_dir = os.path.join(screen_dir, 'data')
    os.makedirs(responses_dir, exist_ok=True)
    os.makedirs(screen_data_dir, exist_ok=True)

    # --- IDENTIFY TARGET SHEET ---
    # We automatically pick the first available sheet. 
    # Change this string if you want a specific sheet (e.g., 'V1_Exc_L2/3')
    available_sheets = data_store.sheets()
    target_sheet_name = available_sheets[0]
    print(f"Exporting data for sheet: {target_sheet_name}")
    
    neuron_ids = data_store.get_sheet_ids(target_sheet_name)
    n_neurons = len(neuron_ids)

    # --- PROCESS DATA ---
    print("Processing Segments...")
    all_binned_spikes = []
    combined_meta = {}
    screen_timestamps = []
    
    # We maintain a continuous time axis across all concatenated trials
    current_time = 0.0
    valid_segment_count = 0
    
    for seg in all_segments:
        # CRITICAL FIX: Filter at the Segment level
        if seg.annotations.get('sheet_name') != target_sheet_name:
            continue
            
        valid_segment_count += 1
        
        # --- 1. Neural Responses ---
        duration = seg.t_stop - seg.t_start
        n_bins = int(duration.rescale('ms').magnitude / bin_width_ms)
        
        # Initialize bin matrix: (Time, Neurons)
        segment_activity = np.zeros((n_bins, n_neurons), dtype=np.float32)
        
        # All spike trains in this segment belong to the target sheet
        spiketrains = seg.spiketrains
        
        # Ensure we map spikes to the correct column (neuron index)
        # We create a map from source_id to column index
        id_to_col = {nid: i for i, nid in enumerate(neuron_ids)}
        
        for st in spiketrains:
            source_id = st.annotations['source_id']
            if source_id in id_to_col:
                col_idx = id_to_col[source_id]
                
                # Shift spikes relative to segment start
                spike_times = (st.times - seg.t_start).rescale('ms').magnitude
                
                # Bin spikes
                counts, _ = np.histogram(spike_times, bins=n_bins, range=(0, duration.rescale('ms').magnitude))
                segment_activity[:, col_idx] = counts
            
        all_binned_spikes.append(segment_activity)

        # --- 2. Visual Stimuli (Screen) ---
        seg_duration_sec = (seg.t_stop - seg.t_start).rescale('s').magnitude
        stimulus_id = seg.annotations['stimulus']
        
        # Retrieve stimulus frames
        if str(stimulus_id) in data_store.sensory_stimulus:
            frames = data_store.sensory_stimulus[str(stimulus_id)]
        else:
            # Fallback for blanks/missing data
            # Use a default size (e.g., 64x64) or try to infer from metadata
            frames = np.zeros((n_bins, 64, 64)) # Assuming 1 frame per bin for simplicity if missing

        # Ensure frames match the time dimension (approx)
        # Experanto expects correlation between frames and neural bins
        # If mismatch, we might need to resize 'frames' or just save as is.
        # Here we save whatever we got.
        
        file_name = f"{valid_segment_count-1:05d}.npy"
        np.save(os.path.join(screen_data_dir, file_name), frames.astype(np.float32))
        
        # Timestamps for this trial's frames
        n_frames = frames.shape[0]
        trial_timestamps = np.linspace(current_time, current_time + seg_duration_sec, n_frames, endpoint=False)
        screen_timestamps.append(trial_timestamps)
        
        # Metadata
        combined_meta[f"{valid_segment_count-1:05d}"] = {
            'modality': 'video' if n_frames > 1 else 'image',
            'image_size': [frames.shape[1], frames.shape[2]],
            'num_frames': int(n_frames),
            'first_frame_idx': 0,
            'stim_type': str(stimulus_id)
        }
        
        current_time += seg_duration_sec

    # --- SAVE FINAL ARRAYS ---
    if not all_binned_spikes:
        raise ValueError(f"No segments found for sheet {target_sheet_name}. Check sheet names in data_store.sheets()")

    full_response_matrix = np.concatenate(all_binned_spikes, axis=0)
    
    # Save Response Data
    np.save(os.path.join(responses_dir, 'data.npy'), full_response_matrix)
    
    # Save Response Metadata
    response_meta = {
        'modality': 'sequence',
        'sampling_rate': 1000.0 / bin_width_ms,
        'start_time': 0.0,
        'end_time': float(full_response_matrix.shape[0] * (bin_width_ms/1000.0)),
        'n_signals': n_neurons,
        'n_timestamps': full_response_matrix.shape[0],
        'unit_ids': [int(id) for id in neuron_ids]
    }
    with open(os.path.join(responses_dir, 'meta.yml'), 'w') as f:
        yaml.dump(response_meta, f)

    # Save Screen Metadata
    with open(os.path.join(screen_dir, 'combined_meta.json'), 'w') as f:
        json.dump(combined_meta, f)
        
    full_timestamps = np.concatenate(screen_timestamps)
    np.save(os.path.join(screen_dir, 'timestamps.npy'), full_timestamps)
    
    print(f"Conversion complete. Processed {valid_segment_count} segments.")

In [12]:
def export_mozaik_responses_to_experanto(mozaik_root, output_root, bin_width_ms, target_sheet_name=None):
    """
    Exports Mozaik neural data to Experanto 'sequence' format.
    
    Args:
        mozaik_root (str): Path to the Mozaik simulation output folder.
        output_root (str): Path to the Experanto dataset root (where 'responses' will be created).
        bin_width_ms (float): The bin size for spike counts. 
                              CRITICAL: Set this to your 'movie_frame_duration'.
        target_sheet_name (str, optional): Name of the sheet to export (e.g., 'V1_Exc_L2/3').
                                           If None, defaults to the first available sheet.
    """
    # 1. Load Mozaik Data
    print(f"Loading DataStore from: {mozaik_root}")
    # We set store_stimuli=False because we don't need to load the heavy visual data
    data_store = PickledDataStore(
        load=True, 
        parameters=ParameterSet({'root_directory': mozaik_root, 'store_stimuli': False})
    )
    
    # 2. Identify Target Sheet
    if target_sheet_name is None:
        target_sheet_name = data_store.sheets()[0]
    
    print(f"Processing sheet: {target_sheet_name}")
    neuron_ids = data_store.get_sheet_ids(target_sheet_name)
    n_neurons = len(neuron_ids)
    
    # Create mapping from neuron ID to column index to ensure consistent ordering
    id_to_col = {nid: i for i, nid in enumerate(neuron_ids)}

    # 3. Process Segments
    # get_segments(ordered=True) ensures we process trials in the order they were presented
    segments = data_store.get_segments(ordered=True)
    all_binned_spikes = []
    
    print(f"Found {len(segments)} segments. Binning spikes at {bin_width_ms}ms resolution...")

    valid_segments = 0
    for seg in segments:
        # Filter: Only process segments belonging to the target sheet
        # Note: Depending on Mozaik version, sheet_name might be in annotations 
        # or implied by the contained neo objects. 
        if seg.annotations.get('sheet_name') != target_sheet_name:
            continue
            
        valid_segments += 1
        
        # Calculate time bins for this segment
        duration = seg.t_stop - seg.t_start
        duration_ms = duration.rescale('ms').magnitude
        n_bins = int(np.round(duration_ms / bin_width_ms))
        
        # Initialize bin matrix: (Time, Neurons)
        segment_activity = np.zeros((n_bins, n_neurons), dtype=np.float32)
        
        # Fill matrix with spike counts
        for st in seg.spiketrains:
            source_id = st.annotations['source_id']
            if source_id in id_to_col:
                col_idx = id_to_col[source_id]
                
                # Shift spike times to be relative to segment start
                spike_times = (st.times - seg.t_start).rescale('ms').magnitude
                
                # Numpy histogram is fast for binning
                counts, _ = np.histogram(spike_times, bins=n_bins, range=(0, duration_ms))
                segment_activity[:, col_idx] = counts
        
        all_binned_spikes.append(segment_activity)

    if valid_segments == 0:
        raise ValueError(f"No segments found for sheet '{target_sheet_name}'. Available sheets: {data_store.sheets()}")

    # 4. Concatenate and Save
    full_response_matrix = np.concatenate(all_binned_spikes, axis=0)
    
    # Create directory
    responses_dir = os.path.join(output_root, 'responses')
    os.makedirs(responses_dir, exist_ok=True)
    
    # Save Data
    npy_path = os.path.join(responses_dir, 'data.npy')
    np.save(npy_path, full_response_matrix)
    print(f"Saved neural data to {npy_path}. Shape: {full_response_matrix.shape}")
    
    # 5. Generate Metadata (meta.yml)
    # This is required by Experanto's SequenceInterpolator
    response_meta = {
        'modality': 'sequence',
        'sampling_rate': 1000.0 / bin_width_ms,
        'start_time': 0.0,
        'end_time': float(full_response_matrix.shape[0] * (bin_width_ms / 1000.0)),
        'n_signals': n_neurons,
        'n_timestamps': full_response_matrix.shape[0],
        'unit_ids': [int(id) for id in neuron_ids],
        # Optional: Add phase shifts if you have 2-photon scan path info
        'phase_shift_per_signal': False 
    }
    
    with open(os.path.join(responses_dir, 'meta.yml'), 'w') as f:
        yaml.dump(response_meta, f)
    
    print("Export Complete.")

# --- Usage Example ---
# If your Mozaik experiment used 'movie_frame_duration': 33.33 (30Hz)
# export_mozaik_responses_to_experanto(
#     mozaik_root='./MozaikOutput', 
#     output_root='./ExperantoDataset', 
#     bin_width_ms=33.33
# )

In [14]:
def convert_mozaik_to_experanto_responses(mozaik_root, output_root, frame_rate=30.0, target_sheet=None):
    """
    Extracts neural data from Mozaik and saves it as an Experanto 'responses' module.
    
    Args:
        mozaik_root (str): Path to the Mozaik simulation output directory.
        output_root (str): Path to the Experanto dataset root (where 'screen' folder is).
        frame_rate (float): Frame rate of the movie used in simulation (Hz). Default 30.0.
        target_sheet (str): Name of the sheet to export (e.g. 'V1_Exc_L2/3'). 
                            If None, selects the first available sheet.
    """
    bin_width_ms = 1000.0 / frame_rate
    print(f"Loading DataStore from {mozaik_root}...")
    print(f"Using bin width: {bin_width_ms:.2f} ms ({frame_rate} Hz)")

    # Load data without loading stimuli (saves memory)
    data_store = PickledDataStore(
        load=True, 
        parameters=ParameterSet({'root_directory': mozaik_root, 'store_stimuli': False})
    )
    
    # 1. Select Sheet
    available_sheets = data_store.sheets()
    if target_sheet is None:
        target_sheet = available_sheets[0]
    
    if target_sheet not in available_sheets:
        raise ValueError(f"Sheet '{target_sheet}' not found. Available: {available_sheets}")
        
    print(f"Exporting responses from sheet: {target_sheet}")
    neuron_ids = data_store.get_sheet_ids(target_sheet)
    n_neurons = len(neuron_ids)
    
    # Create mapping from neuron ID to column index for consistency
    id_to_col = {nid: i for i, nid in enumerate(neuron_ids)}

    # 2. Process Segments
    # MeasurePixelMovieExperanto plays clips sequentially. We must concatenate them.
    segments = data_store.get_segments(ordered=True)
    all_binned_activity = []
    
    print(f"Processing {len(segments)} segments...")
    
    for i, seg in enumerate(segments):
        # Filter for segments belonging to our target sheet
        if seg.annotations.get('sheet_name') != target_sheet:
            continue
            
        # Get duration of this trial
        duration_ms = (seg.t_stop - seg.t_start).rescale('ms').magnitude
        
        # Determine number of bins (frames) in this segment
        # We use round() to handle slight floating point drifts, assuming segments are integer frames
        n_bins = int(np.round(duration_ms / bin_width_ms))
        
        # Initialize matrix: (Time, Neurons)
        segment_activity = np.zeros((n_bins, n_neurons), dtype=np.float32)
        
        # Populate with spike counts
        for st in seg.spiketrains:
            source_id = st.annotations['source_id']
            if source_id in id_to_col:
                col_idx = id_to_col[source_id]
                
                # Shift spike times to be relative to trial start
                spike_times = (st.times - seg.t_start).rescale('ms').magnitude
                
                # Histogram binning
                counts, _ = np.histogram(spike_times, bins=n_bins, range=(0, duration_ms))
                segment_activity[:, col_idx] = counts
        
        all_binned_activity.append(segment_activity)

    if not all_binned_activity:
        raise ValueError("No valid data found. Check sheet name or simulation output.")

    # 3. Concatenate Trials
    # Experanto expects a single continuous sequence for the 'responses' modality
    full_response_matrix = np.concatenate(all_binned_activity, axis=0)
    print(f"Final Data Shape: {full_response_matrix.shape} (Time x Neurons)")

    # 4. Save to Disk
    resp_dir = os.path.join(output_root, 'responses')
    meta_dir = os.path.join(resp_dir, 'meta')
    os.makedirs(meta_dir, exist_ok=True)

    # Save Data
    np.save(os.path.join(resp_dir, 'data.npy'), full_response_matrix)

    # Save Metadata (meta.yml)
    # This config tells Experanto how to interpolate this data
    meta_config = {
        'modality': 'sequence',
        'sampling_rate': frame_rate,
        'start_time': 0.0,
        'end_time': float(full_response_matrix.shape[0] / frame_rate),
        'n_signals': n_neurons,
        'n_timestamps': full_response_matrix.shape[0],
        'unit_ids': [int(nid) for nid in neuron_ids],
        'is_mem_mapped': False
    }
    
    with open(os.path.join(resp_dir, 'meta.yml'), 'w') as f:
        yaml.dump(meta_config, f)

    # 5. Save Statistics (Optional but recommended for Normalization)
    means = np.mean(full_response_matrix, axis=0)
    stds = np.std(full_response_matrix, axis=0)
    # Avoid division by zero in normalization if a neuron never fired
    stds[stds == 0] = 1.0 
    
    np.save(os.path.join(meta_dir, 'means.npy'), means)
    np.save(os.path.join(meta_dir, 'stds.npy'), stds)

    print(f"Export complete. Data saved to: {resp_dir}")

# Usage Example:
# convert_mozaik_to_experanto_responses(
#     mozaik_root='./MozaikOutput', 
#     output_root='./ExperantoDataset', 
#     frame_rate=30.0,
#     target_sheet='V1_Exc_L2/3'
# )

In [10]:
convert_mozaik_to_experanto(mozaik_root='SelfSustainedPushPull_test_____', output_root='screen_test', bin_width_ms=33.33)

Loading Mozaik Datastore...
Exporting data for sheet: X_ON
Processing Segments...
Conversion complete. Processed 28 segments.


In [13]:
export_mozaik_responses_to_experanto(mozaik_root='SelfSustainedPushPull_test_____', output_root='response', bin_width_ms=33.33)

Loading DataStore from: SelfSustainedPushPull_test_____
Processing sheet: X_ON
Found 168 segments. Binning spikes at 33.33ms resolution...
Saved neural data to response/responses/data.npy. Shape: (1200, 7200)
Export Complete.


In [ ]:
convert_mozaik_to_experanto_responses(mozaik_root='SelfSustainedPushPull_test:big32_____', output_root='final_response', frame_rate=30.0, target_sheet='V1_Exc_L2/3')

Loading DataStore from SelfSustainedPushPull_test:big32_____...
Using bin width: 33.33 ms (30.0 Hz)
Exporting responses from sheet: V1_Exc_L2/3
Processing 168 segments...
